# 시장 규모 추출기 일관성 검증

동일 원문에 `temperature=0.8`로 N회 반복 추출 → 필드별 일관성 분석 → AI 채점

**검증 대상:** `section_extractor.extract_section()` (market_size 중심)

---

## 0. 환경 설정

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("."))  # 프로젝트 루트를 경로에 추가

# Django 설정 (모델 사용 시 필요)
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
import django
django.setup()

## 1. 설정

In [2]:
SECTIONS = ["market_size", "kbeauty_share", "trends", "channels", "competitors"]
COUNTRIES = ["US", "JP"]
N_RUNS    = 10
TEMP      = 0.8

print(f"총 {len(SECTIONS) * len(COUNTRIES)}개 조합 × {N_RUNS}회 = {len(SECTIONS) * len(COUNTRIES) * N_RUNS}번 추출")

총 10개 조합 × 10회 = 100번 추출


In [4]:
# 원문 미리보기 (단일 섹션/국가 확인용)
SECTION = "market_size"
COUNTRY = "US"

from market_api.services.section_crawler import crawl_section
raw_text, source_url = crawl_section(SECTION, COUNTRY)
print(raw_text[:2000])

   [US] 시장 규모·성장률 크롤링 중... https://www.zionmarketresearch.com/report/us-skincare-market...
      OK 11,256자 수집
U.S. Skincare Industry Perspective:

The U.S. skincare market size was worth around

USD 25.04 billion

in 2024 and is predicted to grow to around

USD 36.56 billion

by 2034, with a compound annual growth rate

(CAGR)

of roughly

3.86%

between 2025 and 2034.

Request Free Sample

U.S. Skincare Market: Overview

The U.S. skincare industry is one of the most lucrative and highest revenue-generating markets for the global personal care sector. It deals with skincare items such as face moisturizers and body lotions produced or distributed in the US region. The regional industry is filled with a wide range of companies, including domestic brands as well as international entities.

Furthermore, one of the most crucial current trends in the US skincare sector is the increasing launch of celebrity-owned brands that have gained significant growth momentum in the past few years. The 

---
## 2. 전체 섹션 × 국가 검증 실행

In [54]:
import importlib
import json
from datetime import datetime
import market_api.services.extractor_validator as ev
importlib.reload(ev)

from market_api.services.section_crawler import crawl_section

all_results = {}
SAVE_FILE = f"validation_all_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

def save_all():
    output = {
        "meta": {
            "sections": SECTIONS,
            "countries": COUNTRIES,
            "n_runs": N_RUNS,
            "temperature": TEMP,
            "run_at": datetime.now().isoformat(),
        },
        "results": {
            f"{sec}_{ctr}": data
            for (sec, ctr), data in all_results.items()
        },
    }
    with open(SAVE_FILE, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"  💾 저장: {SAVE_FILE} ({len(all_results)}개 완료)")

for section in SECTIONS:
    for country in COUNTRIES:
        key = (section, country)
        print(f"\n{'═' * 55}")
        print(f"  [{section}] {country}")
        print(f"{'═' * 55}")

        raw_text, source_url = crawl_section(section, country)
        if not raw_text:
            print("  ⚠️  원문 없음 — 건너뜀")
            continue

        print(f"  temperature={TEMP}로 {N_RUNS}회 추출 중...")
        results = ev.run_extractions(section, country, raw_text, source_url, N_RUNS, TEMP, max_workers=5)

        analysis = ev.analyze_consistency(results, section)

        print(f"  AI 채점 중...")
        report = ev.ai_score(section, country, N_RUNS, analysis)

        all_results[key] = {
            "extraction_results": results,
            "consistency_analysis": analysis,
            "validation_report": report,
        }

        ev.print_report(report, section, country)
        save_all()  # 섹션 완료마다 저장

print(f"\n\n완료: {len(all_results)}/{len(SECTIONS)*len(COUNTRIES)}개 조합")


═══════════════════════════════════════════════════════
  [market_size] US
═══════════════════════════════════════════════════════
   [US] 시장 규모·성장률 크롤링 중... https://www.zionmarketresearch.com/report/us-skincare-market...
      OK 11,256자 수집
  temperature=0.8로 30회 추출 중...
  [30/30] 완료
총 30개 성공, 0개 실패
  AI 채점 중...

───────────────────────────────────────────────────────
  추출기 검증 보고서 | market_size / US
───────────────────────────────────────────────────────
  accuracy            ██████████  4.90/5.00
  consistency         ██████████  4.90/5.00
  hallucination_free  ██████████  5.00/5.00
  overall             ██████████  4.90/5.00
───────────────────────────────────────────────────────

  [권고사항]
    • 통화·단위 표기를 후처리로 정규화해 billion/Billion 같은 대소문자 차이를 제거하세요.
    • 평가 시 문자열 완전일치 외에 단위·대소문자 정규화 기준을 적용해 실질 일관성을 더 정확히 반영하세요.

  [종합 평가]
  US market_size 추출은 30회 반복에서도 수치 값이 거의 완벽하게 고정되어 매우 신뢰도가 높습니다. value와 forecast_value의 소수 변동도 없고 단지 표기 대소문자 차이만 존재해 정확도, 일관성, 비환각성 모두 최상위 수준입니다.
  💾 저장: validati

In [4]:
# 결과 샘플 확인 (특정 조합)
import json
key = ("market_size", "US")
if key in all_results:
    print(json.dumps(all_results[key]["extraction_results"][0], ensure_ascii=False, indent=2))

NameError: name 'all_results' is not defined

In [3]:
# trends/JP 단일 재검증 (프롬프트 수정 후)
import importlib
import market_api.services.extractor_validator as ev
import market_api.services.section_extractor as se
importlib.reload(se)
importlib.reload(ev)

from market_api.services.section_crawler import crawl_section

_section, _country = "trends", "JP"
raw_text, source_url = crawl_section(_section, _country)

print(f"temperature={TEMP}로 {N_RUNS}회 추출 중...")
_results  = ev.run_extractions(_section, _country, raw_text, source_url, N_RUNS, TEMP, max_workers=5)
_analysis = ev.analyze_consistency(_results, _section)
_report   = ev.ai_score(_section, _country, N_RUNS, _analysis)

ev.print_report(_report, _section, _country)

# all_results 업데이트 후 저장
all_results[(_section, _country)] = {
    "extraction_results": _results,
    "consistency_analysis": _analysis,
    "validation_report": _report,
}
save_all()

   [JP] 성분·기능 트렌드 크롤링 중... https://www.sourceready.com/report/detail/japan-skincare-mar...
      OK 12,384자 수집
temperature=0.8로 10회 추출 중...
  [5] 실패: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
  [4] 실패: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
  [3] 실패: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides

ZeroDivisionError: division by zero

In [10]:
# trends JP 원문 확인
from market_api.services.section_crawler import crawl_section
_raw, _url = crawl_section("trends", "JP")
print(f"총 {len(_raw):,}자\n")
print(_raw)

   [JP] 성분·기능 트렌드 크롤링 중... https://www.sourceready.com/report/detail/japan-skincare-mar...
      OK 12,384자 수집
총 12,384자

Japan Skincare Market Report 2025: Trends, Brands & Products

Executive Summary

The Japanese skincare market stands as one of the world's most sophisticated beauty markets, projected to reach

$12.52 billion by 2033

with a robust growth rate of 4.46% annually

Market Industry Research (note.com)

. In 2025, the market is characterized by a unique fusion of traditional Japanese ingredients with cutting-edge biotechnology, reflecting the nation's commitment to innovation while honoring centuries-old beauty philosophies.

Market Overview and Growth Trajectory

The broader Japanese beauty and personal care market is on a strong growth path, expected to expand from

$32.97 billion in 2025 to $38.76 billion by 2030

, representing a compound annual growth rate of 3.29%

Mordor Intelligence (mordorintelligence.com)

. Within this ecosystem, skincare dominates as the larg

---
## 3. 전체 결과 요약

In [56]:
# 전체 종합 점수 비교표
print(f"{'섹션':<20} {'국가'}  {'overall':>8}  {'accuracy':>9}  {'consistency':>12}  {'hall_free':>10}")
print("─" * 70)
for (section, country), data in all_results.items():
    scores = data["validation_report"].get("scores", {})
    def s(k): return scores.get(k, {}).get("score", 0.0)
    print(f"{section:<20} {country}    {s('overall'):>5.2f}      {s('accuracy'):>5.2f}         {s('consistency'):>5.2f}        {s('hallucination_free'):>5.2f}")

섹션                   국가   overall   accuracy   consistency   hall_free
──────────────────────────────────────────────────────────────────────
market_size          US     4.90       4.90          4.90         5.00
market_size          JP     4.90       4.90          4.80         4.90
kbeauty_share        US     5.00       5.00          5.00         4.90
kbeauty_share        JP     4.80       4.80          4.60         4.90
trends               US     4.50       4.40          4.60         4.70
trends               JP     3.20       3.70          2.80         3.10
channels             US     5.00       5.00          5.00         5.00
channels             JP     4.10       4.00          4.40         3.80
competitors          US     5.00       4.90          5.00         5.00
competitors          JP     4.90       4.80          5.00         5.00


In [57]:
import json
from datetime import datetime

# all_results의 key를 문자열로 변환 (JSON 직렬화용)
serializable = {
    f"{sec}_{ctr}": {
        "consistency_analysis": data["consistency_analysis"],
        "validation_report":    data["validation_report"],
        "extraction_results":   data["extraction_results"],
    }
    for (sec, ctr), data in all_results.items()
}

output = {
    "meta": {
        "sections": SECTIONS,
        "countries": COUNTRIES,
        "n_runs": N_RUNS,
        "temperature": TEMP,
        "run_at": datetime.now().isoformat(),
    },
    "results": serializable,
}

fname = f"validation_all_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(fname, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {fname}")

저장 완료: validation_all_20260328_160811.json
